# Competitive kernel response with GELU: 64 abnormal + 64 normal
Keep LGT and one Conv1D(512,128,k=5). Apply GELU to all responses BEFORE the two channel maxima: logit1 = max(GELU(first 64 channels)) - max(GELU(last 64 channels)) + bias. The group roles are learned from video labels, not snippet labels.
Train from scratch for 20 epochs, LR 5e-6, milestones 8/16, gradient clipping 1.0, deterministic algorithms and independent data seed 234. Only GELU is added; losses and training settings match the latest no-GELU configuration. Evaluate after every epoch for monitoring only; epoch 20 is the primary result, never selected by test AUC. Deterministic execution may be slower and will stop explicitly if an operation is unsupported. Run cells in order.

In [ ]:
import csv
import json
import shlex
import shutil
import subprocess
import sys
from pathlib import Path
import torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
RESULT_ROOT = PROJECT_ROOT / 'code' / 'results'
SCORE_MODE = 'competitive_response_gelu'
RUN_TAG = 'cnn_compete_gelu_s234'
TRAIN_DIR = RESULT_ROOT / 'train' / RUN_TAG
EVAL_DIR = RESULT_ROOT / 'evaluate' / RUN_TAG
ANALYZE_DIR = RESULT_ROOT / 'analyze' / RUN_TAG
EPOCH_DIR = SRC_DIR / 'model' / 'epoch_checkpoints_cnn_single_conv' / RUN_TAG
MODEL_PATH = EPOCH_DIR / 'epoch_020.pth'
CONFIG_PATH = TRAIN_DIR / 'config.json'

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime first.')
if not SRC_DIR.is_dir():
    raise FileNotFoundError(SRC_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archive = next((PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz') if (PROJECT_ROOT / name).is_file()), None)
    if archive is None:
        raise FileNotFoundError('Missing extracted features and feature archive on Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    command = ['unzip', '-q', str(local_archive), '-d', '/content'] if archive.suffix == '.zip' else ['tar', '-xf', str(local_archive), '-C', '/content']
    subprocess.run(command, check=True)
required = [FEATURE_ROOT]
required += [SRC_DIR / name for name in ('model_cnn_vadclip.py', 'ucf_train_cnn_vadclip.py', 'ucf_evaluate_cnn_vadclip.py', 'test_cnn_single_conv.py', 'ucf_cnn_common.py')]
required += [LIST_DIR / name for name in ('ucf_CLIP_rgb_relative.csv', 'ucf_CLIP_rgbtest_relative.csv', 'gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')]
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
print('GPU:', torch.cuda.get_device_name(0), '| Run:', RUN_TAG)
print('Features:', FEATURE_ROOT)

def run_command(command):
    print('Running:', shlex.join(map(str, command)), flush=True)
    with subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait() != 0:
            raise RuntimeError('Command failed; see the traceback above.')

def options_to_args(options):
    argv = []
    for key, value in options.items():
        if isinstance(value, bool):
            if value:
                argv.append('--' + key.replace('_', '-'))
            continue
        argv.append('--' + key.replace('_', '-'))
        argv.extend(str(item) for item in (value if isinstance(value, list) else [value]))
    return argv


## Verify and train
Checks raw signed responses, both-group gradients, padding, parser support and a full CUDA forward/backward before training. A matching completed epoch-20 run skips training; an incomplete matching run restarts from scratch. Per-epoch metrics are printed and saved separately; evaluation adds runtime.

In [ ]:
# Detect outdated uploads before starting a long run.
for name in ('model_cnn_vadclip.py', 'ucf_train_cnn_vadclip.py', 'ucf_evaluate_cnn_vadclip.py', 'test_cnn_single_conv.py'):
    if 'competitive_response_gelu' not in (SRC_DIR / name).read_text(encoding='utf-8'):
        raise RuntimeError(f'Upload the updated {name} to {SRC_DIR} first.')
run_command([sys.executable, '-m', 'unittest', 'test_cnn_single_conv'])
if '--evaluate-every-epoch' not in (SRC_DIR / 'ucf_train_cnn_vadclip.py').read_text(encoding='utf-8'):
    raise RuntimeError('Upload the updated ucf_train_cnn_vadclip.py first.')

options = {
    'feature_root': str(FEATURE_ROOT), 'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv'),
    'cnn_architecture': 'single_conv', 'score_mode': SCORE_MODE,
    'kernel_size': 5, 'cnn_channels': 128, 'aggregation': 'topk',
    'lambda_alignment': 1.0, 'lambda_text': 1.0, 'seed': 234,
    'batch_size': 64, 'lr': 5e-6, 'max_epoch': 20, 'visual_length': 256,
    'scheduler_milestones': [8, 16], 'scheduler_rate': 0.1,
    'data_seed': 234, 'deterministic': True, 'grad_clip_norm': 1.0,
    'evaluate_every_epoch': True,
    'test_list': str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
    'gt_path': str(LIST_DIR / 'gt_ucf.npy'),
    'gt_segment_path': str(LIST_DIR / 'gt_segment_ucf.npy'),
    'gt_label_path': str(LIST_DIR / 'gt_label_ucf.npy'),
    'evaluate_dir': str(EVAL_DIR), 'analyze_dir': str(ANALYZE_DIR),
    'early_stopping_patience': 0, 'num_workers': 2,
    'epoch_checkpoint_dir': str(EPOCH_DIR),
    'output_model_path': str(SRC_DIR / 'model' / 'cnn_single_conv' / f'{RUN_TAG}.pth'),
    'results_dir': str(TRAIN_DIR),
}
if CONFIG_PATH.is_file():
    saved = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    mismatches = [key for key, value in options.items() if saved.get(key) != value]
    if mismatches:
        raise RuntimeError(f'Existing config differs at {mismatches}; choose a new RUN_TAG.')
elif EPOCH_DIR.exists() and any(EPOCH_DIR.glob('*.pth')):
    raise RuntimeError('Checkpoint exists without config; choose a new RUN_TAG.')
if MODEL_PATH.is_file() and (TRAIN_DIR / 'early_stopping.json').is_file():
    print('Matching completed epoch-20 run; skipping train:', MODEL_PATH)
else:
    print('Training from scratch:', SCORE_MODE, '| 64 abnormal + 64 normal kernels, Top-K video pooling.')
    run_command([sys.executable, '-u', str(SRC_DIR / 'ucf_train_cnn_vadclip.py')] + options_to_args(options))


## Read epoch 20 and per-epoch metrics
Reuse the evaluation produced during training; run final evaluation only if missing. Epoch folders preserve every result. Previous kernel-max and Linear results are shown if available; they used the older training schedule and are not a controlled single-factor comparison.

In [ ]:
FINAL_EVAL_DIR = EVAL_DIR / 'epoch_020'
FINAL_ANALYZE_DIR = ANALYZE_DIR / 'epoch_020'
evaluate_options = {
    'model_type': 'cnn', 'model_path': str(MODEL_PATH),
    'cnn_architecture': 'single_conv', 'score_mode': SCORE_MODE,
    'kernel_size': 5, 'cnn_channels': 128, 'train_config_json': str(CONFIG_PATH),
    'feature_root': str(FEATURE_ROOT), 'test_list': str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
    'gt_path': str(LIST_DIR / 'gt_ucf.npy'), 'gt_segment_path': str(LIST_DIR / 'gt_segment_ucf.npy'),
    'gt_label_path': str(LIST_DIR / 'gt_label_ucf.npy'),
    'evaluate_dir': str(FINAL_EVAL_DIR), 'analyze_dir': str(FINAL_ANALYZE_DIR),
}
if not (FINAL_EVAL_DIR / 'metrics.json').is_file() or not (FINAL_ANALYZE_DIR / 'summary.json').is_file():
    run_command([sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_cnn_vadclip.py')] + options_to_args(evaluate_options))
history_path = TRAIN_DIR / 'evaluation_history.csv'
if history_path.is_file():
    print(history_path.read_text(encoding='utf-8'))
metrics = json.loads((FINAL_EVAL_DIR / 'metrics.json').read_text(encoding='utf-8'))
previous_root = RESULT_ROOT / 'evaluate' / 'cnn_single_conv' / 'cnn_single_readout_ablation_s234'
rows = []
keys = ['classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'avg_mAP']
raw_path = RESULT_ROOT / 'evaluate' / 'cnn_compete_stable_s234' / 'epoch_020' / 'metrics.json'
if raw_path.is_file():
    raw_metrics = json.loads(raw_path.read_text(encoding='utf-8'))
    rows.append({'variant': 'competitive_no_gelu (20 epochs)', **{key: raw_metrics[key] for key in keys}})
for name in ('kernel_max', 'linear_readout'):
    path = previous_root / name / 'metrics.json'
    if path.is_file():
        old = json.loads(path.read_text(encoding='utf-8'))
        rows.append({'variant': name + ' (previous)', **{key: old[key] for key in keys}})
rows.append({'variant': SCORE_MODE, **{key: metrics[key] for key in keys}})
print('Variant | AUC1 | AP1 | AUC2 | AP2 | Ano-AUC | mAP')
for row in rows:
    print(row['variant'] + ' | ' + ' | '.join(f'{row[key]:.2f}' for key in keys))
with (EVAL_DIR / 'comparison.csv').open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['variant'] + keys)
    writer.writeheader()
    writer.writerows(rows)
summary = json.loads((FINAL_ANALYZE_DIR / 'summary.json').read_text(encoding='utf-8'))
print('Top-15 anomaly fraction (macro):', summary['selected_anomaly_fraction_abnormal_video_macro'])
for group, values in summary['groups'].items():
    print(group, '| count:', values['count'], '| mean:', values['score_mean'], '| p10:', values['score_p10'], '| p50:', values['score_p50'])
print('Train:', TRAIN_DIR, 'Evaluate:', EVAL_DIR, 'Analyze:', ANALYZE_DIR, sep='\n')


## Phân tích hai nhóm kernel và so sánh trên cùng frame
Chỉ cần chạy cell setup đầu notebook rồi cell này, không chạy lại train. Chọn ANALYSIS_RUN_TAG để phân tích lượt có hoặc không GELU. Analyzer đọc chế độ thực tế từ config của lượt train, không dùng SCORE_MODE hiện tại.
Mặc định phân tích các epoch 1, 4, 8, 10, 20 nếu đã lưu; dùng epoch cuối trong danh sách tìm được để cố định nhóm true-high/false-high/missed anomaly. Vẫn lưu thêm thống kê theo Top-15% riêng của từng checkpoint. Kernel thắng là phép Max theo channel; nếu bằng điểm, lượt thắng được chia đều.
Có thể điền BASELINE_SELF_PATH (chỉ baseline tự train) và LINEAR_MODEL_PATH cùng LINEAR_CONFIG_PATH để so sánh thêm. Nếu không điền, chỉ đối chiếu các checkpoint competitive. Threshold cùng FPR được tính trên test chỉ để chẩn đoán, không dùng khi train hoặc triển khai. Mỗi lần phân tích có thư mục output mới.

In [ ]:
from datetime import datetime
ANALYSIS_RUN_TAG = 'cnn_compete_gelu_s234'
ANALYSIS_EPOCHS = [1, 4, 8, 10, 20]  # Use list(range(1, 21)) for all epochs.
ANALYSIS_CONFIG = RESULT_ROOT / 'train' / ANALYSIS_RUN_TAG / 'config.json'
ANALYSIS_CHECKPOINT_DIR = SRC_DIR / 'model' / 'epoch_checkpoints_cnn_single_conv' / ANALYSIS_RUN_TAG
BASELINE_SELF_PATH = None  # Example: SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth'
LINEAR_MODEL_PATH = None
LINEAR_CONFIG_PATH = None
analyzer = SRC_DIR / 'ucf_analyze_competitive_kernels.py'
for required in (analyzer, ANALYSIS_CONFIG):
    if not required.is_file():
        raise FileNotFoundError(required)
models = []
for epoch in sorted(set(ANALYSIS_EPOCHS)):
    path = ANALYSIS_CHECKPOINT_DIR / f'epoch_{epoch:03d}.pth'
    if path.is_file():
        models.append({'name': f'competitive_e{epoch:03d}', 'model_type': 'cnn', 'epoch': epoch,
                       'model_path': str(path), 'train_config_json': str(ANALYSIS_CONFIG)})
    else:
        print('Missing epoch, skipped:', path)
if not models:
    raise FileNotFoundError('No checkpoints found; check ANALYSIS_CHECKPOINT_DIR and ANALYSIS_EPOCHS.')
reference_model = models[-1]['name']
comparison_reference = reference_model
if len(models) == 1:
    print('Only one checkpoint: cannot measure kernel evolution across epochs.')
if BASELINE_SELF_PATH is not None:
    models.append({'name': 'baseline_self', 'model_type': 'baseline', 'baseline_source': 'self_trained',
                   'model_path': str(BASELINE_SELF_PATH)})
    comparison_reference = 'baseline_self'
if LINEAR_MODEL_PATH is not None:
    if LINEAR_CONFIG_PATH is None:
        raise ValueError('Provide the matching training config for Linear.')
    models.append({'name': 'linear', 'model_type': 'cnn', 'model_path': str(LINEAR_MODEL_PATH),
                   'train_config_json': str(LINEAR_CONFIG_PATH)})
for model in models:
    for key in ('model_path', 'train_config_json'):
        if key in model and not Path(model[key]).is_file():
            raise FileNotFoundError(model[key])
DIAG_DIR = RESULT_ROOT / 'analyze' / 'competitive_kernels' / ANALYSIS_RUN_TAG / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
DIAG_DIR.mkdir(parents=True, exist_ok=False)
models_json = DIAG_DIR / 'models.json'
models_json.write_text(json.dumps({'reference_model': reference_model, 'comparison_reference': comparison_reference,
                                  'models': models}, indent=2), encoding='utf-8')
print('Fixed groups from:', reference_model, '| Score comparison against:', comparison_reference)
print('Checkpoints:', [model['name'] for model in models])
run_command([sys.executable, '-u', str(analyzer), '--models-json', str(models_json),
             '--feature-root', str(FEATURE_ROOT), '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
             '--gt-path', str(LIST_DIR / 'gt_ucf.npy'), '--output-dir', str(DIAG_DIR),
             '--top-k-ratio', '0.15', '--chunk-batch-size', '16', '--expected-videos', '290'])
print('Diagnostics saved:', DIAG_DIR)
